# DocIQ on Kaggle: baseline, training, evaluation
Settings (right sidebar): **Accelerator = GPU T4 x2**, **Internet = On**. Add a Kaggle secret named `HF_TOKEN` (Add-ons > Secrets) with a Hugging Face **write** token.
Run the cells top to bottom. Each stage saves to `/kaggle/working`, so a dropped session costs little: re-run from the failed stage (training supports `--resume`).
Full explanation: `explain/06-runbook-what-is-left.md`.

In [ ]:
!git clone https://github.com/Trilokuday3/DocIQ.git /kaggle/working/DocIQ 2>/dev/null || (cd /kaggle/working/DocIQ && git pull)
%cd /kaggle/working/DocIQ
!pip install -q -e ".[train]" qwen-vl-utils
from kaggle_secrets import UserSecretsClient
import os
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
import torch; print(torch.cuda.get_device_name(0))

## 1. Baseline: zero-shot Qwen on VALIDATION (3 prompts, keep the best)

In [ ]:
for v in ["v1", "v2", "v3"]:
    !python -m dociq.evaluate --kind qwen --model Qwen/Qwen2.5-VL-3B-Instruct --split validation --prompt {v} --name qwen_{v} --mlflow-run baseline-qwen3b-{v}

Look at `reports/qwen_v*.json`. If a run crashes, paste the traceback to Claude (most likely a transformers-version detail in `models/baseline_vlm.py`).

## 2. Train Donut (3-6 h on a T4)
Edit `configs/train_donut.yaml` first: set `hub_model_id: <your-hf-username>/dociq-donut-cord`. After a disconnect re-run this cell with `--resume`.

In [ ]:
!python -m dociq.train --config configs/train_donut.yaml

## 3. Final evaluation on TEST (run once) + robustness + reference checkpoint

In [ ]:
BEST = "outputs/donut/best"   # or your Hub id
!python -m dociq.evaluate --kind donut --model {BEST} --split test --name donut --degrade all
!python -m dociq.evaluate --kind donut --model naver-clova-ix/donut-base-finetuned-cord-v2 --split test --name donut_official_ref
!python -m dociq.evaluate --kind qwen --model Qwen/Qwen2.5-VL-3B-Instruct --split test --prompt v1 --name qwen_test --degrade all   # use your best prompt

In [ ]:
!python -m dociq.report > reports/results.md
!cat reports/results.md
from dociq.report import confidence_table, error_summary
print(confidence_table("reports/donut.jsonl"))
print(error_summary("reports/donut.jsonl"))

## 4. Save everything
Download `reports/` (and `mlflow.db`) from the Output tab, then commit `reports/results.md` to the repo. Latency/cost numbers here are for the T4; measure CPU latency separately (`--quantize` and no GPU).